In [1]:
import pandas as pd
import os,glob,math
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt


In [2]:
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
plt.rc("font",family="Arial")

In [3]:
from statannotations.Annotator import Annotator
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [4]:
# pd.set_option('display.max_columns', 10)
pd.set_option('display.max_rows', 10)

In [5]:
dfall = pd.read_csv("Sst_neurons_sholl.csv",index_col=0)

In [6]:
dfall

,cluster,10.0,20.0,30.0,40.0,50.0,60.0,70.0,80.0,90.0,...,910.0,920.0,930.0,940.0,950.0,960.0,970.0,980.0,990.0,1000.0
221058_038_Sst,Sst_long,6.0,12.0,16.0,17.0,19.0,18.0,15.0,15.0,13.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
221058_107_Sst,Sst_local,3.0,4.0,8.0,8.0,9.0,10.0,12.0,14.0,14.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
221297_038_Sst,Sst_long,2.0,9.0,16.0,18.0,19.0,21.0,22.0,21.0,18.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
252460_132_Sst,Sst_local,7.0,6.0,6.0,6.0,6.0,7.0,8.0,9.0,12.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
221058_040_Sst,Sst_local,6.0,5.0,6.0,6.0,8.0,9.0,10.0,10.0,11.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
252459_074_Sst,Sst_local,6.0,7.0,12.0,14.0,15.0,15.0,15.0,15.0,14.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
221297_037_Sst,Sst_long,5.0,9.0,12.0,16.0,18.0,21.0,18.0,17.0,15.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
230058_073_Sst,Sst_local,6.0,7.0,11.0,14.0,17.0,17.0,19.0,18.0,15.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
221058_088_Sst,Sst_local,2.0,4.0,5.0,5.0,7.0,7.0,8.0,9.0,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
df = dfall[["50.0","60.0","cluster"]]

In [13]:
df.columns = ["50.0","60.0","group"]

In [15]:
from scipy.stats import ttest_ind

group_order = ["Sst_long", "Sst_local"]

for para in ["50.0", "60.0"]:

    # ============================================================
    # Unpaired two-tailed Student's t-test
    # ============================================================
    group1, group2 = group_order

    data1 = pd.to_numeric(
        df.loc[df["group"] == group1, para],
        errors="coerce"
    ).dropna()

    data2 = pd.to_numeric(
        df.loc[df["group"] == group2, para],
        errors="coerce"
    ).dropna()

    t_stat, p_value = ttest_ind(
        data1,
        data2,
        equal_var=True,          # Student's unpaired t-test
        alternative="two-sided"
    )

    # 保存统计结果
    summary = pd.DataFrame({
        "parameter": [para],
        "group1": [group1],
        "group2": [group2],
        "n1": [len(data1)],
        "n2": [len(data2)],
        "t-statistic": [t_stat],
        "p-value": [p_value]
    })

    summary.to_csv(
        f"sholl_results_of_{para}.csv",
        index=False
    )

    # 用于 statannotations
    group_list = [(group1, group2)]
    p_values = [p_value]

    # ============================================================
    # 绘图
    # ============================================================
    plt.figure(figsize=(2.5, 6))

    sns.stripplot(
        data=df,
        x="group",
        y=para,
        order=group_order,
        jitter=0.2,
        edgecolor="black",
        palette=["white", "white"],
        size=9,
        linewidth=1.5
    )

    sns.barplot(
        data=df,
        x="group",
        y=para,
        order=group_order,
        palette=["#2B11EF", "#51A3E1"],
        errorbar="se",
        capsize=0.4,
        errcolor="0",
        errwidth=2,
        edgecolor="0",
        linewidth=1.5
    )

    # ============================================================
    # 添加统计学显著性注释
    # ============================================================
    annotator = Annotator(
        plt.gca(),
        group_list,
        data=df,
        x="group",
        y=para,
        order=group_order
    )

    annotator.configure(
        test=None,
        text_format="simple",
        loc="inside",
        verbose=0,
        fontsize=16,
        pvalue_format_string="{:.3f}",
        line_width=2
    )

    annotator.set_pvalues(p_values).annotate()

    # ============================================================
    # 图形格式
    # ============================================================
    plt.xlabel("")
    plt.ylabel(para, fontsize=16)

    plt.tick_params(axis="x", labelsize=16)
    plt.xticks(rotation=30, ha="right", va="top")

    plt.tick_params(axis="y", labelsize=16)

    plt.tight_layout()

    plt.savefig(f"Sholl_sst_{para}.pdf", dpi=300)
    plt.savefig(f"Sholl_sst_{para}.jpg", dpi=300)

    plt.close()

C:\Users\zljia\AppData\Local\Temp\ipykernel_25668\2178176200.py:54: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(
C:\Users\zljia\AppData\Local\Temp\ipykernel_25668\2178176200.py:66: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
C:\Users\zljia\AppData\Local\Temp\ipykernel_25668\2178176200.py:66: FutureWarning: 

The `errcolor` parameter is deprecated. And will be removed in v0.15.0. Pass `err_kws={'color': '0'}` instead.

  sns.barplot(
C:\Users\zljia\AppData\Local\Temp\ipykernel_25668\2178176200.py:66: FutureWarning: 

The `errwidth` parameter is deprecated. And will be removed in v0.15.0. Pass `err_kws={'linewidth': 2}` instead.

  sns.barplot(
C:\Users\zljia\AppData\Local\Temp\ipyker